In [1]:
pip install arabizikit

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("rabeeeehh/arabizi-kit-corpus")

def load_split(name):
    e = ds[name][0]["entries"]
    # handles both list-of-dicts and dict-of-lists
    return pd.DataFrame(e)

tr, va, te = load_split("train"), load_split("validation"), load_split("test")
print(len(tr), len(va), len(te))
print(tr.columns.tolist())
print(tr["dialect"].value_counts())

eg = tr[tr["dialect"] == "egyptian"].rename(columns={"reference": "ar"})
df = eg[["arabizi", "ar"]].drop_duplicates("arabizi").reset_index(drop=True)
print(len(df))
print(df.head())

226 48 49
['id', 'arabizi', 'reference', 'dialect', 'note']
dialect
egyptian     122
msa           78
other         18
levantine      6
gulf           1
maghrebi       1
Name: count, dtype: int64
122
                                             arabizi  \
0  laken el 5obz el mosatta7 elly geh ma3 el beid...   
1               el 7odoor el naharda beya3kes e7sasy   
2  ga3an lama mesheet talabt el ceviche fel 3ard ...   
3            7abeeby konna notlob sandwiches classic   
4                                  ma2boul el takyif   

                                                  ar  
0           لكن الخبز المسطح اللي جاي مع البيض بتاعي  
1                           الحضور اليوم يعكس إحساسي  
2  جان لما مشيت طلبت السيفيتشي في العرض الخاص ب29...  
3                    حبيبي كنا نطلب ساندويتشز كلاسيك  
4                                      مقبول التكييف  


In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

# 1) Small subset (Test 256 and 32. Maybe other shit Idk.)
sub = df.sample(32, random_state=0)
pairs = list(zip(sub["arabizi"], sub["ar"]))

PAD, SOS, EOS = 0, 1, 2
# PAD: shorter words get padded by 0.
# SOS: Start of sequenece.
# EOS: End of sequence.

def make_vocab(strs):
    return {c: i + 3 for i, c in enumerate(sorted(set("".join(strs))))} #concat every words into one large string.
src_v = make_vocab([a for a, _ in pairs])  #
tgt_v = make_vocab([b for _, b in pairs])
inv_tgt = {i: c for c, i in tgt_v.items()}


def pad(seqs):
    L = max(map(len, seqs))
    return torch.tensor([s + [PAD] * (L - len(s)) for s in seqs])

X = pad([[src_v[c] for c in a] for a, _ in pairs])
Y = pad([[SOS] + [tgt_v[c] for c in b] + [EOS] for _, b in pairs])

# 2) Model: 4 LSTM layers, no dropout, no regularization
class Seq2Seq(nn.Module):
    def __init__(s, nsrc, ntgt, emb=64, hid=256, layers=4):
        super().__init__()
        s.se = nn.Embedding(nsrc, emb, padding_idx=PAD)
        s.te = nn.Embedding(ntgt, emb, padding_idx=PAD)
        s.enc = nn.LSTM(emb, hid, layers, batch_first=True)
        s.dec = nn.LSTM(emb, hid, layers, batch_first=True)
        s.out = nn.Linear(hid, ntgt)
    def forward(s, x, y_in):
        _, st = s.enc(s.se(x))
        o, _ = s.dec(s.te(y_in), st)
        return s.out(o)

V = len(tgt_v) + 3
m = Seq2Seq(len(src_v) + 3, V)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss(ignore_index=PAD)

# 3) Train full-batch on the same data you evaluate on
for ep in range(1, 601):
    logits = m(X, Y[:, :-1])
    loss = lossf(logits.reshape(-1, V), Y[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step()
    if ep % 50 == 0: print(ep, round(loss.item(), 4))

# 4) Exact-match check with greedy decoding
@torch.no_grad()
def greedy(x, maxlen=30):
    _, st = m.enc(m.se(x))
    tok = torch.full((x.size(0), 1), SOS)
    outs = []
    for _ in range(maxlen):
        o, st = m.dec(m.te(tok), st)
        tok = m.out(o).argmax(-1)
        outs.append(tok)
    return torch.cat(outs, 1).tolist()

def to_str(ids):
    s = []
    for i in ids:
        if i == EOS: break
        if i > 2: s.append(inv_tgt[i])
    return "".join(s)

preds = [to_str(p) for p in greedy(X)]
print("train exact match:", sum(p == b for p, (_, b) in zip(preds, pairs)) / len(pairs))

50 3.0319
100 2.7318
150 2.3138
200 1.9354
250 1.4615
300 0.9144
350 0.6023
400 0.2642
450 0.1178
500 0.071
550 0.0561
600 0.0367
train exact match: 0.125


In [11]:
import matplotlib.pyplot as plt

losses, accs = [], []
for ep in range(1, 601):
    logits = m(X, Y[:, :-1])
    tgt = Y[:, 1:]
    loss = lossf(logits.reshape(-1, V), tgt.reshape(-1))
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step()

    with torch.no_grad():
        mask = tgt != PAD
        acc = ((logits.argmax(-1) == tgt) & mask).sum().item() / mask.sum().item()
    losses.append(loss.item()); accs.append(acc)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(losses); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(accs);   ax[1].set_title("Train char accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

KeyboardInterrupt: 